# BCF-1 — Protected late rank fusion

Bounded diagnostic over frozen `DEV_CROSS_60` plus fresh `DEV_L21_150`. F1 copies A0 Top5 exactly and applies equal RRF60 only to the remaining final A0/S1 Top100 candidates. The exact prebuilt SigLIP2 index is mandatory and is never rebuilt. Every A0/S1/F1 prediction is finalized and hashed before GT is opened; no production policy is promoted automatically.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "0") == "1"
DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_EVAL_INPUT = Path(os.environ.get("AIC_TEAM_EVAL_DEV_ROOT", "/kaggle/input/datasets/irthn1311/aic2026_team_eval_dev_v1"))
STAGE1_INPUT = Path(os.environ.get("AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"))
STAGE1B_INPUT = Path(os.environ.get("AIC_STAGE1B_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports"))
STAGE1E_INPUT = Path(os.environ.get("AIC_STAGE1E_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze"))
CLIP_INPUT = Path(os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"))
OPUS_INPUT = Path(os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en"))
SIGLIP_INPUT = Path(os.environ.get("AIC_SIGLIP2_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-siglip2-base-patch16-224"))
FREEZE_INPUT = Path(os.environ.get("AIC_BCF1_FREEZE_ROOT", "/kaggle/input/datasets/irthn1311/bcf1-preparation-freeze-2026-08-18"))
INDEX_INPUT = Path(os.environ.get("AIC_SCA1_INDEX_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-sca1-siglip2-index-v01"))
SIGLIP_DEVICE = os.environ.get("AIC_SIGLIP2_DEVICE", "auto")
SIGLIP_BATCH_SIZE = int(os.environ.get("AIC_SIGLIP2_BATCH_SIZE", "64"))
OUTPUT_ROOT = Path("/kaggle/working/artifacts/bcf1_protected_late_fusion_v01")
WORK_ROOT = Path("/kaggle/working/triage_eg_bcf1_work")
ZIP_PATH = Path("/kaggle/working/triage_eg_bcf1_protected_late_fusion_v01_bundle.zip")
for cleanup_target in (OUTPUT_ROOT, WORK_ROOT):
    if cleanup_target.exists():
        if Path("/kaggle/working") not in cleanup_target.parents:
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {cleanup_target}")
        shutil.rmtree(cleanup_target)
ZIP_PATH.unlink(missing_ok=True)
print({
    "required_inputs": {
        "raw_dataset": str(DATA_INPUT),
        "team_eval_dev_bundle_cross_and_l21": str(TEAM_EVAL_INPUT),
        "stage1_exact_index": str(STAGE1_INPUT),
        "stage1b_verified_contract": str(STAGE1B_INPUT),
        "stage1e_language_contract": str(STAGE1E_INPUT),
        "openai_clip_offline_asset": str(CLIP_INPUT),
        "opus_mt_vi_en_offline_asset": str(OPUS_INPUT),
        "siglip2_offline_asset": str(SIGLIP_INPUT),
        "bcf1_preparation_freeze": str(FREEZE_INPUT),
        "exact_prebuilt_sca1_siglip2_index": str(INDEX_INPUT),
    },
    "internet_required": "ONLY_FOR_GIT_CLONE_OR_EXPLICIT_REFRESH",
    "model_download_required": False,
    "siglip2_index_policy": "REQUIRED_PREBUILT_EXACT_INDEX; REBUILD_FORBIDDEN",
    "output_zip": str(ZIP_PATH),
})


In [ ]:
def git_result(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)

def git(*args, cwd=None):
    result = git_result(*args, cwd=cwd)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR))
target_ref = None
if not REFRESH_REPO:
    for candidate in (REPO_REF, f"origin/{REPO_REF}"):
        probe = git_result("rev-parse", "--verify", f"{candidate}^{{commit}}", cwd=REPO_DIR)
        if probe.returncode == 0:
            target_ref = probe.stdout.strip()
            break
if target_ref is None:
    git("fetch", "--no-tags", "origin", REPO_REF, cwd=REPO_DIR)
    target_ref = "FETCH_HEAD"
git("checkout", "--detach", target_ref, cwd=REPO_DIR)
HEAD = git("rev-parse", "HEAD", cwd=REPO_DIR)
SOURCE_REF = REPO_REF
CHECKOUT_MODE = "DETACHED_PINNED_REF"
anchor = "c26d5b1f8ffe96557387ffdf2b8beb6d134bfefa"
anchor_probe = git_result("cat-file", "-e", f"{anchor}^{{commit}}", cwd=REPO_DIR)
if anchor_probe.returncode != 0:
    git("fetch", "--no-tags", "origin", REPO_REF, cwd=REPO_DIR)
    anchor_probe = git_result("cat-file", "-e", f"{anchor}^{{commit}}", cwd=REPO_DIR)
if anchor_probe.returncode != 0:
    raise RuntimeError(f"SCA-1 anchor commit unavailable after ref fetch: {anchor}")
ancestor = git_result("merge-base", "--is-ancestor", anchor, HEAD, cwd=REPO_DIR).returncode == 0
if not ancestor:
    raise RuntimeError(f"SCA-1 anchor is not an ancestor of {HEAD}: {anchor}")
module_file = REPO_DIR / "src/triage_eg/diagnostics/bcf1_protected_late_fusion/runner.py"
if not module_file.is_file():
    raise RuntimeError("Resolved TRIAGEEG ref does not contain reviewed BCF-1 source")
sys.path.insert(0, str(REPO_DIR / "src"))
GIT_STATUS = git("status", "--short", cwd=REPO_DIR)
BRANCH = SOURCE_REF
print({
    "source_ref": SOURCE_REF,
    "HEAD": HEAD,
    "checkout_mode": CHECKOUT_MODE,
    "git_status": GIT_STATUS or "CLEAN",
    "sca1_anchor_is_ancestor": ancestor,
})


In [ ]:
MAX_DEPTH, MAX_DIRECTORIES = 5, 2048
SPECIAL_SLUG_ALIASES = {
    "dataset-aic": ("dataset-aic2026", "Dataset_AIC2026"),
    "aic2026_team_eval_dev_v1": ("aic2026-team-eval-dev-v1",),
    "bcf1-preparation-freeze-2026-08-18": ("BCF1_PREPARATION_FREEZE_2026-08-18",),
}

def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError(f"Input discovery exceeded {MAX_DIRECTORIES} directories below {root}")
        yield current
        if depth < MAX_DEPTH:
            queue.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir() and not child.is_symlink()
            )

def named_mount_roots(hint):
    hint = Path(hint)
    slugs = [hint.name, hint.name.replace("_", "-"), hint.name.replace("-", "_")]
    slugs.extend(SPECIAL_SLUG_ALIASES.get(hint.name, ()))
    candidates = [hint]
    for slug in dict.fromkeys(slugs):
        candidates.extend((
            Path("/kaggle/input") / slug,
            Path("/kaggle/input/datasets/irthn1311") / slug,
            Path("/kaggle/input/datasets/nadkli") / slug,
        ))
    return sorted({path.resolve() for path in candidates if path.exists()})

def resolve_mount(hint):
    roots = named_mount_roots(hint)
    if len(roots) != 1:
        raise RuntimeError(f"Expected exactly one mounted dataset for {hint}; found {roots}")
    return roots[0]

def find_marker(root, marker, *, kind="file"):
    marker = Path(marker)
    for directory in bounded_dirs(root):
        candidate = directory / marker
        if (kind == "file" and candidate.is_file()) or (kind == "dir" and candidate.is_dir()):
            return directory
    return None

def resolve_root(hint, marker, *, optional=False):
    matches = []
    for root in named_mount_roots(hint):
        match = find_marker(root, marker)
        if match is not None:
            matches.append(match.resolve())
    matches = sorted(set(matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one root with {marker} for {hint}; found {matches}")
    return matches[0]

def resolve_file(hint, filename, *, optional=False):
    matches = []
    for root in named_mount_roots(hint):
        parent = find_marker(root, filename)
        if parent is not None:
            matches.append((parent / filename).resolve())
    matches = sorted(set(matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {filename} for {hint}; found {matches}")
    return matches[0]

def resolve_dataset(hint):
    marker = "map-keyframes-aic25-b1/map-keyframes"
    matches = []
    for root in named_mount_roots(hint):
        for directory in bounded_dirs(root):
            if (directory / marker).is_dir() and any(directory.glob("Videos_*/video")):
                matches.append(directory.resolve())
                break
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one raw dataset root for {hint}; found {matches}")
    return matches[0]

DATASET_ROOT = resolve_dataset(DATA_INPUT)
TEAM_EVAL_ROOT_MOUNT = resolve_root(
    TEAM_EVAL_INPUT, "benchmarks/dev_cross_60/queries.jsonl", optional=True
)
TEAM_EVAL_ZIP_MOUNT = (
    None if TEAM_EVAL_ROOT_MOUNT
    else resolve_file(TEAM_EVAL_INPUT, "aic2026_team_eval_dev_v1.zip")
)
if TEAM_EVAL_ROOT_MOUNT and not (
    TEAM_EVAL_ROOT_MOUNT / "benchmarks/dev_l21_150/queries.jsonl"
).is_file():
    raise RuntimeError("TEAM-EVAL mount lacks DEV_L21_150 queries")
FREEZE_ROOT_MOUNT = resolve_root(
    FREEZE_INPUT, "bcf1_preparation/decision_context.json", optional=True
)
FREEZE_ZIP_MOUNT = (
    None if FREEZE_ROOT_MOUNT
    else resolve_file(FREEZE_INPUT, "BCF1_PREPARATION_FREEZE_2026-08-18.zip")
)
FREEZE_SOURCE = FREEZE_ROOT_MOUNT or FREEZE_ZIP_MOUNT
STAGE1_MOUNT = resolve_mount(STAGE1_INPUT)
STAGE1B_MOUNT = resolve_mount(STAGE1B_INPUT)
STAGE1E_MOUNT = resolve_mount(STAGE1E_INPUT)
CLIP_MOUNT = resolve_mount(CLIP_INPUT)
OPUS_MOUNT = resolve_mount(OPUS_INPUT)
SIGLIP_MOUNT = resolve_mount(SIGLIP_INPUT)
INDEX_ROOT_MOUNT = resolve_root(INDEX_INPUT, "index/siglip2_vectors.f16.npy", optional=True)
INDEX_ZIP_MOUNT = resolve_file(
    INDEX_INPUT, "triage_eg_sca1_siglip2_index_v01.zip", optional=True
)
if INDEX_ROOT_MOUNT is None and INDEX_ZIP_MOUNT is None:
    raise RuntimeError("Exact prebuilt SCA-1 SigLIP2 index is required; rebuilding is forbidden")
print({
    "resolved_raw": str(DATASET_ROOT),
    "resolved_team_eval_zip": str(TEAM_EVAL_ZIP_MOUNT) if TEAM_EVAL_ZIP_MOUNT else None,
    "resolved_team_eval_root": str(TEAM_EVAL_ROOT_MOUNT) if TEAM_EVAL_ROOT_MOUNT else None,
    "resolved_bcf1_freeze": str(FREEZE_SOURCE),
    "resolved_stage1_mount": str(STAGE1_MOUNT),
    "resolved_stage1b_mount": str(STAGE1B_MOUNT),
    "resolved_stage1e_mount": str(STAGE1E_MOUNT),
    "resolved_clip_mount": str(CLIP_MOUNT),
    "resolved_opus_mount": str(OPUS_MOUNT),
    "resolved_siglip2_mount": str(SIGLIP_MOUNT),
    "resolved_prebuilt_index_root": str(INDEX_ROOT_MOUNT) if INDEX_ROOT_MOUNT else None,
    "resolved_prebuilt_index_zip": str(INDEX_ZIP_MOUNT) if INDEX_ZIP_MOUNT else None,
})


In [ ]:
import yaml
from aic2026_eval.io import sha256_file
from triage_eg.diagnostics.bcf1_protected_late_fusion import (
    BCF1Settings,
    load_preparation_freeze,
    validate_frozen_index,
)
from triage_eg.diagnostics.bcf1_protected_late_fusion.contracts import INDEX_ZIP_SHA256
from triage_eg.diagnostics.sca1_siglip2_complementarity import validate_offline_asset
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root

MATERIALIZED = {
    name: WORK_ROOT / name
    for name in ("stage1", "stage1b", "stage1e", "clip", "opus", "siglip2")
}
STAGE1_ROOT = resolve_stage1_root(
    STAGE1_MOUNT, search_root=None, materialize_root=MATERIALIZED["stage1"]
)
STAGE1B_ROOT, _ = resolve_input_root(
    STAGE1B_MOUNT,
    required=("stage1b_summary.json", "encoder/selected_encoder_contract.json", "encoder/runtime_adapter_manifest.json"),
    materialize_root=MATERIALIZED["stage1b"], search_root=None, archive_keyword="stage1b",
)
STAGE1E_ROOT, _ = resolve_input_root(
    STAGE1E_MOUNT,
    required=("stage1e_summary.json", "language_path_contract.json"),
    materialize_root=MATERIALIZED["stage1e"], search_root=None, archive_keyword="stage1e",
)
CLIP_ROOT, _ = resolve_input_root(
    CLIP_MOUNT,
    required=("checkpoint/ViT-B-32.pt", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED["clip"], search_root=None, archive_keyword="clip",
)
OPUS_ROOT, _ = resolve_input_root(
    OPUS_MOUNT,
    required=("model/config.json", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED["opus"], search_root=None, archive_keyword="opus",
)
SIGLIP_ROOT, _ = resolve_input_root(
    SIGLIP_MOUNT,
    required=("model/model.safetensors", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED["siglip2"], search_root=None, archive_keyword="siglip2",
)
SETTINGS = BCF1Settings()
EXPERIMENT_CONFIG = yaml.safe_load(
    (REPO_DIR / "configs/experiments/triage_bcf1_protected_late_fusion_v01.yaml").read_text(encoding="utf-8")
)
YAML_CHECKS = {
    "experiment": (EXPERIMENT_CONFIG.get("experiment"), "TRIAGE_BCF1_PROTECTED_LATE_FUSION"),
    "scope": (EXPERIMENT_CONFIG.get("scope"), "DIAGNOSTIC_ONLY"),
    "policy.name": (EXPERIMENT_CONFIG.get("policy", {}).get("name"), SETTINGS.policy),
    "policy.a0_protected_prefix": (EXPERIMENT_CONFIG.get("policy", {}).get("a0_protected_prefix"), SETTINGS.protected_prefix),
    "policy.rrf_k": (EXPERIMENT_CONFIG.get("policy", {}).get("rrf_k"), SETTINGS.rrf_k),
    "policy.max_predictions": (EXPERIMENT_CONFIG.get("policy", {}).get("max_predictions"), SETTINGS.max_predictions),
    "index.rebuild_forbidden": (EXPERIMENT_CONFIG.get("index", {}).get("rebuild_forbidden"), True),
    "production_policy_changed": (EXPERIMENT_CONFIG.get("production_policy_changed"), False),
}
YAML_MISMATCHES = {
    field: {"actual": actual, "expected": expected}
    for field, (actual, expected) in YAML_CHECKS.items() if actual != expected
}
if YAML_MISMATCHES:
    raise RuntimeError("BCF1 YAML/dataclass contract mismatch: " + json.dumps(YAML_MISMATCHES, sort_keys=True))
PREPARATION = load_preparation_freeze(FREEZE_SOURCE)
ASSET_VALIDATION = validate_offline_asset(SIGLIP_ROOT)
if INDEX_ROOT_MOUNT is None:
    if sha256_file(INDEX_ZIP_MOUNT) != INDEX_ZIP_SHA256:
        raise RuntimeError("BCF1 exact prebuilt index ZIP hash mismatch")
    INDEX_EXTRACT_ROOT = WORK_ROOT / "prebuilt_siglip2_index"
    INDEX_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with ZipFile(INDEX_ZIP_MOUNT) as archive:
        archive.extractall(INDEX_EXTRACT_ROOT)
    INDEX_ROOT = find_marker(INDEX_EXTRACT_ROOT, "index/siglip2_vectors.f16.npy")
    if INDEX_ROOT is None:
        raise RuntimeError("Prebuilt SigLIP2 index ZIP lacks required index marker")
else:
    INDEX_ROOT = INDEX_ROOT_MOUNT
INDEX_VALIDATION = validate_frozen_index(
    INDEX_ROOT, stage1_root=STAGE1_ROOT, index_zip=INDEX_ZIP_MOUNT
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({
    "freeze_validation": PREPARATION.validation,
    "siglip2_asset_validation": ASSET_VALIDATION,
    "index_validation": INDEX_VALIDATION,
    "index_built_this_run": False,
    "post_gt_design_sanity_opened": False,
    "kaggle_working_free_bytes": shutil.disk_usage("/kaggle/working").free,
})


In [ ]:
TEST_COMMAND = [
    sys.executable, "-m", "pytest",
    "tests/unit/bcf1_protected_late_fusion",
    "tests/integration/test_bcf1_protected_late_fusion.py",
    "tests/unit/sca1_siglip2_complementarity",
    "tests/integration/test_sca1_siglip2_complementarity.py",
    "tests/unit/e2eg1", "tests/unit/e2e1", "tests/unit/stage2_runtime", "-q",
]
TEST_ENV = dict(os.environ)
test_python_paths = [str(REPO_DIR / "src")]
if TEST_ENV.get("PYTHONPATH"):
    test_python_paths.append(TEST_ENV["PYTHONPATH"])
TEST_ENV["PYTHONPATH"] = os.pathsep.join(test_python_paths)
TEST_ENV["AIC_BCF1_FREEZE_SOURCE"] = str(FREEZE_SOURCE)
import_probe = subprocess.run(
    [sys.executable, "-c", (
        "import json, triage_eg, aic2026_eval; "
        "print(json.dumps({'triage_eg': triage_eg.__file__, "
        "'aic2026_eval': aic2026_eval.__file__}, sort_keys=True))"
    )],
    cwd=REPO_DIR, env=TEST_ENV, capture_output=True, text=True, check=False,
)
if import_probe.returncode != 0:
    raise RuntimeError("BCF1 child-process import probe failed: " + (import_probe.stderr.strip() or import_probe.stdout.strip()))
TEST_IMPORT_ORIGINS = json.loads(import_probe.stdout)
expected_source_root = (REPO_DIR / "src").resolve()
for package, origin in TEST_IMPORT_ORIGINS.items():
    if expected_source_root not in Path(origin).resolve().parents:
        raise RuntimeError(f"BCF1 child-process package origin mismatch for {package}: {origin}")
test_process = subprocess.run(
    TEST_COMMAND, cwd=REPO_DIR, env=TEST_ENV, capture_output=True, text=True, check=False
)
test_text = (test_process.stdout + "\n" + test_process.stderr).strip()
match = re.search(r"(\d+) passed", test_text)
TEST_SUMMARY = {
    "command": " ".join(TEST_COMMAND),
    "pythonpath": TEST_ENV["PYTHONPATH"],
    "import_origins": TEST_IMPORT_ORIGINS,
    "returncode": test_process.returncode,
    "passed": int(match.group(1)) if match else None,
    "status": "PASS" if test_process.returncode == 0 else "FAIL",
    "output_tail": test_text.splitlines()[-50:],
}
print(TEST_SUMMARY)
if test_process.returncode != 0:
    print("BCF1_PYTEST_FULL_OUTPUT_BEGIN")
    print(test_text)
    print("BCF1_PYTEST_FULL_OUTPUT_END")
    raise RuntimeError("BCF1 required tests failed before experiment")


In [ ]:
from triage_eg.diagnostics.bcf1_protected_late_fusion.io import write_jsonl_lf
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    Siglip2OfflineEncoder,
    local_only_load_smoke,
)

def read_benchmark_queries(member):
    if TEAM_EVAL_ZIP_MOUNT:
        with ZipFile(TEAM_EVAL_ZIP_MOUNT) as archive:
            lines = archive.read(member).decode("utf-8").splitlines()
    else:
        lines = (TEAM_EVAL_ROOT_MOUNT / member).read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line]

CROSS_QUERY_ROWS = read_benchmark_queries("benchmarks/dev_cross_60/queries.jsonl")
L21_QUERY_ROWS = read_benchmark_queries("benchmarks/dev_l21_150/queries.jsonl")
CROSS_QUERY_ONLY_ROOT = WORK_ROOT / "inference_only/dev_cross_60"
L21_QUERY_ONLY_ROOT = WORK_ROOT / "inference_only/dev_l21_150"
for query_root, rows in ((CROSS_QUERY_ONLY_ROOT, CROSS_QUERY_ROWS), (L21_QUERY_ONLY_ROOT, L21_QUERY_ROWS)):
    query_root.mkdir(parents=True, exist_ok=True)
    write_jsonl_lf(query_root / "queries.jsonl", rows)
    if {path.name for path in query_root.iterdir()} != {"queries.jsonl"}:
        raise RuntimeError(f"BCF1 queries-only boundary failed: {query_root}")
LOCAL_ONLY_SMOKE = local_only_load_smoke(SIGLIP_ROOT)
SIGLIP_ENCODER = Siglip2OfflineEncoder(
    SIGLIP_ROOT, device=SIGLIP_DEVICE, batch_size=SIGLIP_BATCH_SIZE
).load()
print({
    "cross_queries": len(CROSS_QUERY_ROWS),
    "l21_queries": len(L21_QUERY_ROWS),
    "GT_OPENED": False,
    "local_only_siglip2_smoke": LOCAL_ONLY_SMOKE,
    "index_reused": str(INDEX_ROOT),
    "index_rebuilt": False,
})


In [ ]:
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    Siglip2ExactBackend,
    Siglip2GroundingPipeline,
)
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.retrieval.stage1b.adapters.openai_clip_official import (
    materialize_kaggle_expanded_tokenizer,
    resolve_official_asset_paths,
)
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, config_from_yaml

CLIP_ASSET_PATHS = resolve_official_asset_paths(CLIP_ROOT)
SHARED_CLIP_SOURCE_ROOT, SHARED_CLIP_SOURCE_MATERIALIZED = materialize_kaggle_expanded_tokenizer(
    CLIP_ASSET_PATHS.source_root, WORK_ROOT / "shared_openai_clip_source"
)
os.environ["AIC_OPENAI_CLIP_SOURCE_ROOT"] = str(SHARED_CLIP_SOURCE_ROOT)
def make_runtime(name):
    config = config_from_yaml(
        REPO_DIR / "configs/retrieval/stage2_operational_runtime_gpu.yaml",
        stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT,
        clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT,
        output_root=WORK_ROOT / f"runtime_{name}",
        stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
        build_git_commit=HEAD,
    )
    return OperationalRetrievalRuntime(config).load()
A0_RUNTIME = make_runtime("a0")
S1_RUNTIME = make_runtime("s1")
SIGLIP_BACKEND = Siglip2ExactBackend(INDEX_ROOT, stage1_root=STAGE1_ROOT)
A0_PIPELINE = SafeCoveragePipeline(A0_RUNTIME, DATASET_ROOT)
S1_PIPELINE = Siglip2GroundingPipeline(
    S1_RUNTIME, DATASET_ROOT,
    grounding_encoder=SIGLIP_ENCODER, grounding_backend=SIGLIP_BACKEND,
)
print({
    "GT_AVAILABLE_TO_PREDICTION": False,
    "runtime_identity_distinct": A0_RUNTIME is not S1_RUNTIME,
    "pipeline_identity_distinct": A0_PIPELINE is not S1_PIPELINE,
    "final_list_fusion_only": True,
    "production_runtime_mutated": False,
})


In [ ]:
from triage_eg.diagnostics.bcf1_protected_late_fusion import (
    fuse_l21,
    reproduce_cross,
    run_l21_arm,
    validate_all_hashes_before_gt,
)

CROSS_RUN = reproduce_cross(
    PREPARATION, CROSS_QUERY_ROWS, OUTPUT_ROOT, settings=SETTINGS
)
L21_A0_RUN = run_l21_arm(
    A0_PIPELINE, L21_QUERY_ONLY_ROOT, OUTPUT_ROOT, WORK_ROOT / "prediction_temp", "A0"
)
L21_S1_RUN = run_l21_arm(
    S1_PIPELINE, L21_QUERY_ONLY_ROOT, OUTPUT_ROOT, WORK_ROOT / "prediction_temp", "S1"
)
L21_RUN = fuse_l21(L21_A0_RUN, L21_S1_RUN, OUTPUT_ROOT, settings=SETTINGS)
INTEGRITY = validate_all_hashes_before_gt(CROSS_RUN, L21_RUN, INDEX_VALIDATION)
print({
    "prediction_hashes": INTEGRITY["prediction_hashes"],
    "cross_f1_reproduction_gate": INTEGRITY["cross_f1_reproduction_gate"],
    "l21_all_hashes_finalized_before_gt": INTEGRITY["l21_all_hashes_finalized_before_gt"],
    "siglip2_index_reused_without_rebuild": INTEGRITY["siglip2_index_reused_without_rebuild"],
    "GT_OPENED": False,
})


In [ ]:
from triage_eg.diagnostics.bcf1_protected_late_fusion import (
    evaluate_post_gt,
    load_post_gt_design_sanity,
    promotion_decision,
)
from triage_eg.e2e1 import extract_development_bundle

if INTEGRITY.get("status") != "PASS":
    raise RuntimeError("BCF1 refuses to open GT before every integrity/hash gate passes")
if TEAM_EVAL_ZIP_MOUNT:
    TEAM_EVAL_ZIP = TEAM_EVAL_ZIP_MOUNT
    TEAM_EVAL_ROOT = extract_development_bundle(
        TEAM_EVAL_ZIP, WORK_ROOT / "team_eval_extracted"
    )
else:
    TEAM_EVAL_ZIP = None
    TEAM_EVAL_ROOT = TEAM_EVAL_ROOT_MOUNT
CROSS_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_cross_60"
L21_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_l21_150"
for benchmark_root in (CROSS_ROOT, L21_ROOT):
    if not (benchmark_root / "gt.jsonl").is_file():
        raise RuntimeError(f"GT missing after authorized post-hash boundary: {benchmark_root}")
POST_GT_DESIGN_SANITY = load_post_gt_design_sanity(
    PREPARATION, finalized_cross_f1_sha256=CROSS_RUN["F1"]["sha256"]
)
EVALUATIONS = evaluate_post_gt(
    CROSS_RUN, L21_RUN, INTEGRITY,
    cross_gt_path=CROSS_ROOT / "gt.jsonl",
    l21_gt_path=L21_ROOT / "gt.jsonl",
    output_root=OUTPUT_ROOT,
)
DECISION = promotion_decision(INTEGRITY, EVALUATIONS)
print({
    "GT_OPENED_AFTER_ALL_SIX_HASHES": True,
    "post_gt_design_sanity_loaded": True,
    "classification": DECISION["classification"],
    "automatic_production_promotion": DECISION["automatic_production_promotion"],
})


In [ ]:
from triage_eg.diagnostics.bcf1_protected_late_fusion import (
    create_bundle,
    formal_report,
    write_manifests,
)

RESOLVED_INPUTS = {
    "raw_dataset": str(DATASET_ROOT),
    "team_eval_dev_bundle": str(TEAM_EVAL_ZIP or TEAM_EVAL_ROOT),
    "stage1": str(STAGE1_ROOT),
    "stage1b": str(STAGE1B_ROOT),
    "stage1e": str(STAGE1E_ROOT),
    "clip": str(CLIP_ROOT),
    "opus": str(OPUS_ROOT),
    "siglip2_asset": str(SIGLIP_ROOT),
    "bcf1_freeze": str(FREEZE_SOURCE),
    "sca1_siglip2_index": str(INDEX_ROOT),
    "sca1_siglip2_index_zip": str(INDEX_ZIP_MOUNT) if INDEX_ZIP_MOUNT else None,
}
write_manifests(
    OUTPUT_ROOT,
    settings=SETTINGS,
    preparation=PREPARATION,
    index_validation=INDEX_VALIDATION,
    integrity=INTEGRITY,
    cross=CROSS_RUN,
    l21=L21_RUN,
    evaluations=EVALUATIONS,
    decision=DECISION,
    post_gt_design_sanity=POST_GT_DESIGN_SANITY,
    test_summary=TEST_SUMMARY,
    config_snapshot={"yaml": EXPERIMENT_CONFIG, "resolved_inputs": RESOLVED_INPUTS},
    git_commit=HEAD,
    source_ref=SOURCE_REF,
)
BUNDLE = create_bundle(OUTPUT_ROOT, ZIP_PATH)
print(formal_report(
    head=HEAD, integrity=INTEGRITY, evaluations=EVALUATIONS,
    decision=DECISION, bundle=BUNDLE,
))
print("INPUTS_USED=", RESOLVED_INPUTS)
print("DOWNLOAD_ZIP=", ZIP_PATH)
print("INDEX_REBUILD_OCCURRED=", False)
print("PRODUCTION_PROMOTED=", False)
A0_PIPELINE.close()
S1_PIPELINE.close()
SIGLIP_ENCODER.close()
